In [2]:
import numpy as np 
import pandas as pd 
import math
# I want to simulate the algorithm I want to implement in systemverilog.

# I want to load a list of x numbers in a 1D array, similar to how i would load weights into a mem file
weight_rows = 4
weight_cols = 6
num_weights = weight_rows * weight_cols
sys_arr_dim = 3

tile_rows = math.ceil(weight_rows / sys_arr_dim)
tile_cols = math.ceil(weight_cols / sys_arr_dim)


weights_1d = np.random.randint(num_weights, size=num_weights)
weights_matrix = weights_1d.reshape(weight_rows, weight_cols)
print("Weights Matrix:")
print(weights_matrix)

print("1D Weights:")
print(weights_1d)

# I want to tile the weights into 3x3 tiles, similar to how i would tile weights for a systolic array
# Read in Top Left Quadrant, Top Right Quadrant, Bottom Left Quadrant, Bottom Right Quadrant
tiled_weights = []
tile_row_ctr = 0
tile_col_ctr = 0
while(tile_row_ctr < tile_rows):
    tile_col_ctr = 0
    while (tile_col_ctr < tile_cols): # Making sure we don't tile more than needed
        mat_row_ctr = tile_row_ctr * sys_arr_dim # mat_row_ctr is the row index in the original weight matrix
        while(mat_row_ctr < (tile_row_ctr * sys_arr_dim) + sys_arr_dim): # Making sure we don't read more than needed
            # while(tile_col_ctr < tile_cols):
            # mat_col_ctr = tile_col_ctr * weight_cols # mat_col_ctr is the column index in the 
            mat_col_ctr = tile_col_ctr * sys_arr_dim
            while(mat_col_ctr < (tile_col_ctr * sys_arr_dim) + sys_arr_dim):
                addrs = mat_row_ctr * weight_cols + mat_col_ctr
                if(addrs > num_weights - 1): # Making sure we don't read out of bounds
                    tiled_weights.append(0) # Pad with 0 if we read out of bounds
                else:
                    tiled_weights.append(weights_1d[addrs]) # Load One Weight at a time from the 1D array
                mat_col_ctr += 1
            mat_row_ctr += 1
        tile_col_ctr += 1
        print("Tiled Weights:")
        print(np.array(tiled_weights).reshape(sys_arr_dim, sys_arr_dim)) # Print the tiled weights in a 3x3 format
        tiled_weights = [] # Clear the tiled weights for the next tile
    tile_row_ctr += 1



Weights Matrix:
[[ 3 16 21  7 21  4]
 [13 16  4  1 15  4]
 [21 17  6 19 16 16]
 [ 7  3 22 20  5  3]]
1D Weights:
[ 3 16 21  7 21  4 13 16  4  1 15  4 21 17  6 19 16 16  7  3 22 20  5  3]
Tiled Weights:
[[ 3 16 21]
 [13 16  4]
 [21 17  6]]
Tiled Weights:
[[ 7 21  4]
 [ 1 15  4]
 [19 16 16]]
Tiled Weights:
[[ 7  3 22]
 [ 0  0  0]
 [ 0  0  0]]
Tiled Weights:
[[20  5  3]
 [ 0  0  0]
 [ 0  0  0]]


In [ ]:
import numpy as np
import math

inp_vec = np.array([[1, 2, 3, 4, 5],
                    [1, 2, 3, 4, 5],
                    [1, 2, 3, 4, 5],
                    [1, 2, 3, 4, 5]]) # 4x5

weights = np.array([[0, 1, 2, 3], # 5x4
                    [4, 5, 6, 7],
                    [8, 9, 10, 11],
                    [12, 13, 14, 15],
                    [16, 17, 18, 19]]) # 5x4

SYS_ARR_DIM = 4
K_DIM = 5 # The inner dimension shared by inp_vec (cols) and weights (rows)
num_tiles = math.ceil(K_DIM / SYS_ARR_DIM) # Ceiling division ensures we get 2 tiles
result_tile = 0
for i in range(num_tiles):
    # 1. Initialize empty 4x4 tiles with zeros (Handles the padding automatically)
    inp_tile = np.zeros((SYS_ARR_DIM, SYS_ARR_DIM))
    weight_tile = np.zeros((SYS_ARR_DIM, SYS_ARR_DIM))
    
    # 2. Calculate the start and end indices for this specific tile
    start_idx = i * SYS_ARR_DIM
    end_idx = min(start_idx + SYS_ARR_DIM, K_DIM) # min() prevents reading out of bounds
    chunk_size = end_idx - start_idx # Will be 4 on the first loop, and 1 on the second loop
    
    # 3. Extract the real data chunks
    inp_chunk = inp_vec[:, start_idx:end_idx]      # Take all rows, slice the columns
    weight_chunk = weights[start_idx:end_idx, :]   # Slice the rows, take all columns
    
    # 4. Map the extracted chunks into our zero-padded tiles
    inp_tile[:, :chunk_size] = inp_chunk
    weight_tile[:chunk_size, :] = weight_chunk
    
    print(f"--- Tile {i} ---")
    print("Input Tile (4x4):")
    print(inp_tile)
    print("\nWeight Tile (4x4):")
    print(weight_tile)
    print("="*30)

    result_tile += inp_tile @ weight_tile
    print(f"Result after processing Tile {i}:")
    print(result_tile)
    #15 + 

--- Tile 0 ---
Input Tile (4x4):
[[1. 2. 3. 4.]
 [1. 2. 3. 4.]
 [1. 2. 3. 4.]
 [1. 2. 3. 4.]]

Weight Tile (4x4):
[[ 0.  1.  2.  3.]
 [ 4.  5.  6.  7.]
 [ 8.  9. 10. 11.]
 [12. 13. 14. 15.]]
Result after processing Tile 0:
[[ 80.  90. 100. 110.]
 [ 80.  90. 100. 110.]
 [ 80.  90. 100. 110.]
 [ 80.  90. 100. 110.]]
--- Tile 1 ---
Input Tile (4x4):
[[5. 0. 0. 0.]
 [5. 0. 0. 0.]
 [5. 0. 0. 0.]
 [5. 0. 0. 0.]]

Weight Tile (4x4):
[[16. 17. 18. 19.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]
 [ 0.  0.  0.  0.]]
Result after processing Tile 1:
[[160. 175. 190. 205.]
 [160. 175. 190. 205.]
 [160. 175. 190. 205.]
 [160. 175. 190. 205.]]
